# 02 · Preprocesamiento del texto (Fase II)

Efecto de cada paso de `clasificador_ml.preprocesamiento` sobre el vocabulario del conjunto de entrenamiento (sección 6.2 del diseño).

Requisito: haber ejecutado `python -m clasificador_ml.datos` y `python -m clasificador_ml.preprocesamiento`.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

from clasificador_ml.preprocesamiento import Preprocesador, normalizar
from clasificador_ml.rutas import DATOS_PROCESADOS

train = pd.read_parquet(DATOS_PROCESADOS / 'preprocesado' / 'train.parquet')
train[['texto', 'texto_procesado']].head(3)

## Vocabulario tras cada paso

Se usa una muestra de 5.000 resúmenes para que los pasos sin caché (sin lematizar) sean rápidos.

In [ ]:
muestra = train.sample(5000, random_state=42)

pasos = {
    '0. Texto original (split)': [t.split() for t in muestra['texto']],
    '1-2. Normalizado': [normalizar(t).split() for t in muestra['texto']],
    '3. Sin stopwords': [t.split() for t in Preprocesador(lematizar=False).transform(muestra['texto'])],
    '4. Lematizado (final)': [t.split() for t in muestra['texto_procesado']],
}
resumen = pd.DataFrame({
    'vocabulario': {p: len(set().union(*map(set, d))) for p, d in pasos.items()},
    'tokens_por_documento': {p: sum(map(len, d)) / len(d) for p, d in pasos.items()},
})
resumen

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
resumen['vocabulario'].plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_xlabel('Términos distintos (muestra de 5.000)')
ax.set_title('Reducción del vocabulario por paso')
plt.tight_layout()

## Ejemplo: un resumen antes y después

In [ ]:
fila = train.iloc[0]
print(fila['texto'][:600], '\n')
print(fila['texto_procesado'][:600])

## Términos más frecuentes por dominio

Si el preprocesamiento funciona, los términos frecuentes de cada dominio deberían ser propios de su disciplina (y no palabras vacías).

In [ ]:
for dominio, grupo in train.groupby('dominio'):
    conteo = Counter(w for t in grupo['texto_procesado'] for w in t.split())
    print(f"{dominio:13s}", ', '.join(w for w, _ in conteo.most_common(12)))

## Documentos que quedan casi vacíos

La API rechaza textos con menos de 20 términos útiles (`TEXTO_INSUFICIENTE`). ¿Cuántos resúmenes del dataset estarían en ese caso?

In [ ]:
terminos = train['texto_procesado'].str.split().str.len()
print(terminos.describe())
print('Con menos de 20 términos:', (terminos < 20).sum())